# create 1 file

In [1]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime, timedelta

wb = Workbook()
ws = wb.active
ws.title = "Invoice"

ws.column_dimensions['A'].width = 3
ws.column_dimensions['B'].width = 12
ws.column_dimensions['C'].width = 12
ws.column_dimensions['D'].width = 12
ws.column_dimensions['E'].width = 12
ws.column_dimensions['F'].width = 12
ws.column_dimensions['G'].width = 3

TITLE_FONT  = Font(name='Calibri', size=24, bold=True, color='FFFFFF')
TITLE_FILL  = PatternFill(start_color='1F3864', end_color='1F3864', fill_type='solid')
SEC_FONT    = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
SEC_FILL    = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
LABEL_FONT  = Font(name='Calibri', size=10, bold=True)
VALUE_FONT  = Font(name='Calibri', size=10)
TOTAL_FONT  = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
TOTAL_FILL  = PatternFill(start_color='1F3864', end_color='1F3864', fill_type='solid')

BORDER = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
C = Alignment(horizontal='center', vertical='center', wrap_text=True)
L = Alignment(horizontal='left',   vertical='top',    wrap_text=True)
R = Alignment(horizontal='right',  vertical='center')

# ── Title ──────────────────────────────────────────────────────────────────────
ws.row_dimensions[2].height = 35
ws.merge_cells('B2:F2')
ws['B2'].value     = 'INVOICE'
ws['B2'].font      = TITLE_FONT
ws['B2'].fill      = TITLE_FILL
ws['B2'].alignment = C

# ── Company Info (left) ───────────────────────────────────────────────────────
ws.merge_cells('B4:D4')
ws['B4'].value     = 'PT TEKNOLOGI MAJU'
ws['B4'].font      = Font(name='Calibri', size=11, bold=True)
ws['B4'].alignment = L

for r, text in enumerate([
    'Jl. Sudirman No. 123',
    'Jakarta Pusat, 12345',
    'Email: info@teknologimaju.com',
    'Phone: (021) 1234-5678'
], start=5):
    ws[f'B{r}'].value = text
    ws[f'B{r}'].font  = VALUE_FONT

# ── Invoice Metadata (right) ──────────────────────────────────────────────────
meta = [
    ('Invoice No:', 'INV-2024-00152'),
    ('Date:', datetime.now().strftime('%d %B %Y')),
    ('Due Date:', (datetime.now() + timedelta(days=30)).strftime('%d %B %Y')),
    ('Payment Status:', 'PENDING'),
]
for i, (label, value) in enumerate(meta, start=4):
    ws[f'E{i}'].value     = label
    ws[f'E{i}'].font      = LABEL_FONT
    ws[f'E{i}'].alignment = L
    ws[f'F{i}'].value     = value
    ws[f'F{i}'].font      = Font(name='Calibri', size=10,
                                  bold=(label == 'Payment Status:'),
                                  color='FF0000' if label == 'Payment Status:' else '000000')
    ws[f'F{i}'].alignment = L

# ── Bill To ───────────────────────────────────────────────────────────────────
ws.row_dimensions[10].height = 22
ws.merge_cells('B10:F10')
ws['B10'].value     = 'BILL TO'
ws['B10'].font      = SEC_FONT
ws['B10'].fill      = SEC_FILL
ws['B10'].alignment = C

bill_to = [
    ('Client Name:', 'PT MITRA BISNIS Indonesia'),
    ('Address:',     'Jl. Gatot Subroto No. 999, Bandung 40123'),
    ('Contact:',     'Budi Santoso | budi@mitraindonesia.com | (022) 9876-5432'),
]
for i, (label, value) in enumerate(bill_to, start=11):
    ws[f'B{i}'].value     = label
    ws[f'B{i}'].font      = LABEL_FONT
    ws.merge_cells(f'C{i}:F{i}')
    ws[f'C{i}'].value     = value
    ws[f'C{i}'].font      = VALUE_FONT
    ws[f'C{i}'].alignment = L

# ── Line Items Header ─────────────────────────────────────────────────────────
ws.row_dimensions[15].height = 20
for col, heading in zip('BCDEF', ['Description', 'Qty', 'Unit Price', 'Tax Rate', 'Total']):
    cell           = ws[f'{col}15']
    cell.value     = heading
    cell.font      = SEC_FONT
    cell.fill      = SEC_FILL
    cell.border    = BORDER
    cell.alignment = C

# ── Line Items Data ───────────────────────────────────────────────────────────
items = [
    ('Professional Consulting Services',  40, 500000),
    ('Software Development (3 months)',    1, 5000000),
    ('System Integration & Testing',      20, 400000),
]
for i, (desc, qty, price) in enumerate(items, start=16):
    ws.row_dimensions[i].height = 30

    ws[f'B{i}'].value = desc;  ws[f'B{i}'].font = VALUE_FONT; ws[f'B{i}'].alignment = L; ws[f'B{i}'].border = BORDER
    ws[f'C{i}'].value = qty;   ws[f'C{i}'].font = VALUE_FONT; ws[f'C{i}'].alignment = C; ws[f'C{i}'].border = BORDER; ws[f'C{i}'].number_format = '#,##0'
    ws[f'D{i}'].value = price; ws[f'D{i}'].font = VALUE_FONT; ws[f'D{i}'].alignment = R; ws[f'D{i}'].border = BORDER; ws[f'D{i}'].number_format = '$#,##0.00'
    ws[f'E{i}'].value = 0.1;   ws[f'E{i}'].font = VALUE_FONT; ws[f'E{i}'].alignment = C; ws[f'E{i}'].border = BORDER; ws[f'E{i}'].number_format = '0.0%'
    ws[f'F{i}'].value = f'=C{i}*D{i}*(1+E{i})'; ws[f'F{i}'].font = VALUE_FONT; ws[f'F{i}'].alignment = R; ws[f'F{i}'].border = BORDER; ws[f'F{i}'].number_format = '$#,##0.00'

# ── Totals ────────────────────────────────────────────────────────────────────
for r, label, formula in [
    (20, 'Subtotal:',  '=SUMPRODUCT(C16:C18,D16:D18)'),
    (21, 'Tax (10%):', '=SUMPRODUCT(C16:C18,D16:D18,E16:E18)'),
    (22, 'TOTAL DUE:', '=F20+F21'),
]:
    ws.row_dimensions[r].height = 24 if r == 22 else 18
    ws[f'D{r}'].value     = label
    ws[f'D{r}'].font      = TOTAL_FONT if r == 22 else LABEL_FONT
    ws[f'D{r}'].fill      = TOTAL_FILL if r == 22 else PatternFill()
    ws[f'D{r}'].alignment = R
    if r == 22: ws[f'D{r}'].border = BORDER
    ws[f'F{r}'].value         = formula
    ws[f'F{r}'].font          = TOTAL_FONT if r == 22 else VALUE_FONT
    ws[f'F{r}'].fill          = TOTAL_FILL if r == 22 else PatternFill()
    ws[f'F{r}'].alignment     = R
    ws[f'F{r}'].border        = BORDER
    ws[f'F{r}'].number_format = '$#,##0.00'

# ── Payment Terms & Notes ─────────────────────────────────────────────────────
ws.row_dimensions[25].height = 22
ws.merge_cells('B25:F25')
ws['B25'].value     = 'PAYMENT TERMS & NOTES'
ws['B25'].font      = SEC_FONT
ws['B25'].fill      = SEC_FILL
ws['B25'].alignment = C

ws.row_dimensions[26].height = 50
ws.merge_cells('B26:F26')
ws['B26'].value = (
    '• Payment terms: Net 30 days from invoice date\n'
    '• Please transfer to Bank Mandiri, Account: 123-456-789\n'
    '• Thank you for your business!'
)
ws['B26'].font      = VALUE_FONT
ws['B26'].alignment = Alignment(horizontal='left', vertical='top', wrap_text=True)
ws['B26'].border    = BORDER

ws.row_dimensions[28].height = 20
ws.merge_cells('B28:F28')
ws['B28'].value     = 'This is an electronically generated invoice. No signature required.'
ws['B28'].font      = Font(name='Calibri', size=9, italic=True, color='666666')
ws['B28'].alignment = C

wb.save('invoice_example.xlsx')
print("Done! Saved to invoice_example.xlsx")

Done! Saved to invoice_example.xlsx


# Looping file

In [ ]:
import calendar
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from datetime import date, timedelta
import random
import os

# Config
YEAR = 2026
OUTPUT_DIR = 'invoices'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Styles
TITLE_FONT = Font(name='Calibri', size=24, bold=True, color='FFFFFF')
TITLE_FILL = PatternFill(start_color='1F3864', end_color='1F3864', fill_type='solid')
SEC_FONT   = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
SEC_FILL   = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
LABEL_FONT = Font(name='Calibri', size=10, bold=True)
VALUE_FONT = Font(name='Calibri', size=10)
TOTAL_FONT = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
TOTAL_FILL = PatternFill(start_color='1F3864', end_color='1F3864', fill_type='solid')

BORDER = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'),  bottom=Side(style='thin')
)
C = Alignment(horizontal='center', vertical='center', wrap_text=True)
L = Alignment(horizontal='left',   vertical='top',    wrap_text=True)
R = Alignment(horizontal='right',  vertical='center')

# Item pool
ITEM_POOL = [
    ('Professional Consulting Services', 500000),
    ('Software Development',             5000000),
    ('System Integration & Testing',     400000),
    ('UI/UX Design Services',            750000),
    ('Data Analysis & Reporting',        600000),
    ('Cloud Infrastructure Setup',      1200000),
    ('API Development',                  900000),
    ('QA & Testing Services',            350000),
    ('Technical Documentation',          250000),
    ('Project Management',               450000),
]

CLIENTS = [
    ('PT MITRA BISNIS Indonesia',  'Jl. Gatot Subroto No. 999, Bandung 40123', 'Budi Santoso | budi@mitraindonesia.com'),
    ('CV SOLUSI DIGITAL Nusantara','Jl. Pemuda No. 55, Surabaya 60271',        'Andi Wijaya | andi@solusid.co.id'),
    ('PT KARYA INOVASI Tbk',       'Jl. Thamrin No. 1, Jakarta 10310',         'Siti Rahayu | siti@karyainovasi.com'),
    ('UD BERKAH JAYA',             'Jl. Raya Bogor No. 88, Depok 16413',       'Hendra Putra | hendra@berkahjaya.id'),
]

# Invoice builder
def build_invoice(inv_date: date, inv_number: str, client: tuple, items: list):
    wb = Workbook()
    ws = wb.active
    ws.title = "Invoice"

    ws.column_dimensions['A'].width = 3
    ws.column_dimensions['B'].width = 14
    ws.column_dimensions['C'].width = 10
    ws.column_dimensions['D'].width = 14
    ws.column_dimensions['E'].width = 10
    ws.column_dimensions['F'].width = 16
    ws.column_dimensions['G'].width = 3

    # Title
    ws.row_dimensions[2].height = 35
    ws.merge_cells('B2:F2')
    ws['B2'].value = 'INVOICE'; ws['B2'].font = TITLE_FONT; ws['B2'].fill = TITLE_FILL; ws['B2'].alignment = C

    # Company info
    ws.merge_cells('B4:D4')
    ws['B4'].value = 'PT TEKNOLOGI MAJU'; ws['B4'].font = Font(name='Calibri', size=11, bold=True); ws['B4'].alignment = L
    for r, text in enumerate(['Jl. Sudirman No. 123', 'Jakarta Pusat, 12345',
                               'Email: info@teknologimaju.com', 'Phone: (021) 1234-5678'], start=5):
        ws[f'B{r}'].value = text; ws[f'B{r}'].font = VALUE_FONT

    # Metadata
    due_date = inv_date + timedelta(days=30)
    meta = [
        ('Invoice No:',     inv_number),
        ('Date:',           inv_date.strftime('%d %B %Y')),
        ('Due Date:',       due_date.strftime('%d %B %Y')),
        ('Payment Status:', 'PENDING'),
    ]
    for i, (label, value) in enumerate(meta, start=4):
        ws[f'E{i}'].value = label; ws[f'E{i}'].font = LABEL_FONT; ws[f'E{i}'].alignment = L
        ws[f'F{i}'].value = value
        ws[f'F{i}'].font  = Font(name='Calibri', size=10,
                                  bold=(label == 'Payment Status:'),
                                  color='FF0000' if label == 'Payment Status:' else '000000')
        ws[f'F{i}'].alignment = L

    # Bill To
    client_name, client_addr, client_contact = client
    ws.row_dimensions[10].height = 22
    ws.merge_cells('B10:F10')
    ws['B10'].value = 'BILL TO'; ws['B10'].font = SEC_FONT; ws['B10'].fill = SEC_FILL; ws['B10'].alignment = C
    for i, (label, value) in enumerate([('Client Name:', client_name),
                                         ('Address:',     client_addr),
                                         ('Contact:',     client_contact)], start=11):
        ws[f'B{i}'].value = label; ws[f'B{i}'].font = LABEL_FONT
        ws.merge_cells(f'C{i}:F{i}')
        ws[f'C{i}'].value = value; ws[f'C{i}'].font = VALUE_FONT; ws[f'C{i}'].alignment = L

    # Line Items Header
    ws.row_dimensions[15].height = 20
    for col, heading in zip('BCDEF', ['Description', 'Qty', 'Unit Price', 'Tax Rate', 'Total']):
        cell = ws[f'{col}15']
        cell.value = heading; cell.font = SEC_FONT; cell.fill = SEC_FILL; cell.border = BORDER; cell.alignment = C

    # Line Items Data
    for idx, (desc, qty, price) in enumerate(items, start=16):
        ws.row_dimensions[idx].height = 28
        ws[f'B{idx}'].value = desc;  ws[f'B{idx}'].font = VALUE_FONT; ws[f'B{idx}'].alignment = L; ws[f'B{idx}'].border = BORDER
        ws[f'C{idx}'].value = qty;   ws[f'C{idx}'].font = VALUE_FONT; ws[f'C{idx}'].alignment = C; ws[f'C{idx}'].border = BORDER; ws[f'C{idx}'].number_format = '#,##0'
        ws[f'D{idx}'].value = price; ws[f'D{idx}'].font = VALUE_FONT; ws[f'D{idx}'].alignment = R; ws[f'D{idx}'].border = BORDER; ws[f'D{idx}'].number_format = '$#,##0.00'
        ws[f'E{idx}'].value = 0.11;  ws[f'E{idx}'].font = VALUE_FONT; ws[f'E{idx}'].alignment = C; ws[f'E{idx}'].border = BORDER; ws[f'E{idx}'].number_format = '0.0%'
        ws[f'F{idx}'].value = f'=C{idx}*D{idx}*(1+E{idx})'; ws[f'F{idx}'].font = VALUE_FONT; ws[f'F{idx}'].alignment = R; ws[f'F{idx}'].border = BORDER; ws[f'F{idx}'].number_format = '$#,##0.00'

    last_item_row = 15 + len(items)

    # Totals
    sub_row   = last_item_row + 2
    tax_row   = sub_row + 1
    total_row = tax_row + 1
    for r, label, formula in [
        (sub_row,   'Subtotal:',  f'=SUMPRODUCT(C16:C{last_item_row},D16:D{last_item_row})'),
        (tax_row,   'Tax (11%):', f'=SUMPRODUCT(C16:C{last_item_row},D16:D{last_item_row},E16:E{last_item_row})'),
        (total_row, 'TOTAL DUE:', f'=F{sub_row}+F{tax_row}'),
    ]:
        ws.row_dimensions[r].height = 24 if r == total_row else 18
        ws[f'D{r}'].value = label
        ws[f'D{r}'].font  = TOTAL_FONT if r == total_row else LABEL_FONT
        ws[f'D{r}'].fill  = TOTAL_FILL if r == total_row else PatternFill()
        ws[f'D{r}'].alignment = R
        if r == total_row: ws[f'D{r}'].border = BORDER
        ws[f'F{r}'].value         = formula
        ws[f'F{r}'].font          = TOTAL_FONT if r == total_row else VALUE_FONT
        ws[f'F{r}'].fill          = TOTAL_FILL if r == total_row else PatternFill()
        ws[f'F{r}'].alignment     = R
        ws[f'F{r}'].border        = BORDER
        ws[f'F{r}'].number_format = '$#,##0.00'

    # Notes
    notes_hdr = total_row + 3
    ws.row_dimensions[notes_hdr].height = 22
    ws.merge_cells(f'B{notes_hdr}:F{notes_hdr}')
    ws[f'B{notes_hdr}'].value = 'PAYMENT TERMS & NOTES'; ws[f'B{notes_hdr}'].font = SEC_FONT; ws[f'B{notes_hdr}'].fill = SEC_FILL; ws[f'B{notes_hdr}'].alignment = C

    notes_row = notes_hdr + 1
    ws.row_dimensions[notes_row].height = 50
    ws.merge_cells(f'B{notes_row}:F{notes_row}')
    ws[f'B{notes_row}'].value = (
        '• Payment terms: Net 30 days from invoice date\n'
        '• Please transfer to Bank Mandiri, Account: 123-456-789\n'
        '• Thank you for your business!'
    )
    ws[f'B{notes_row}'].font = VALUE_FONT; ws[f'B{notes_row}'].alignment = L; ws[f'B{notes_row}'].border = BORDER

    footer_row = notes_row + 2
    ws.row_dimensions[footer_row].height = 20
    ws.merge_cells(f'B{footer_row}:F{footer_row}')
    ws[f'B{footer_row}'].value = 'This is an electronically generated invoice. No signature required.'
    ws[f'B{footer_row}'].font  = Font(name='Calibri', size=9, italic=True, color='666666')
    ws[f'B{footer_row}'].alignment = C

    return wb

# Main loop - one invoice per day for every month of the year
random.seed(42)

for month in range(1, 13):
    days_in_month = calendar.monthrange(YEAR, month)[1]

    for day in range(1, days_in_month + 1):
        inv_date = date(YEAR, month, day)
        inv_number = f'INV-{YEAR}{month:02d}{day:02d}-{day:03d}'
        client = random.choice(CLIENTS)

        n_items = random.randint(2, 4)
        sampled = random.sample(ITEM_POOL, n_items)
        items = [(desc, random.randint(1, 50), price) for desc, price in sampled]

        wb = build_invoice(inv_date, inv_number, client, items)
        wb.save(f'{OUTPUT_DIR}/invoice_{inv_date.strftime("%Y-%m-%d")}.xlsx')